# Experiment 3 — Prompted vs. unprompted BERT-large comparison

This notebook does not re-encode sentences. It **loads the saved embeddings** from Experiments 1 and 2 and compares prompted versus unprompted BERT-large representations: cluster quality (for example silhouette), inter- vs. intra-class distances, and PCA views. Run it after both embedding notebooks have written their results under `results/llm/`.

- The first code cell installs packages from `requirements.txt` only if they are missing.
- Outputs have been cleared from the repository version.


In [ ]:
# Install dependencies if needed, then set repository paths and seeds.
# Standard library only until packages from requirements.txt are present.
from pathlib import Path
import os
import sys
import subprocess
import random

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "llm_data").exists() and (candidate / "llms").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the cloned repository.")

REPO_ROOT = find_repo_root()

REQUIRED_MODULES = (
    "numpy", "pandas", "matplotlib", "seaborn", "scipy", "sklearn",
    "PIL", "openpyxl", "torch", "torchvision", "transformers",
    "accelerate", "huggingface_hub",
)

def dependencies_present():
    import importlib
    for name in REQUIRED_MODULES:
        try:
            importlib.import_module(name)
        except ImportError:
            return False
    return True

req_file = REPO_ROOT / "requirements.txt"
if dependencies_present():
    print("Dependencies already installed; skipping pip.")
else:
    print(f"Installing packages from {req_file} (first run can take several minutes)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)])
    print("Dependencies ready.")

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = REPO_ROOT / "data" / "llm_data"
RESULTS_ROOT = REPO_ROOT / "results" / "llm"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"LLM data:        {DATA_DIR}")


In [ ]:
# ============================================================
# Prompted vs Unprompted — BERT-large-uncased
# ============================================================
import os
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import cdist
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt





DIR_UNPR = str(RESULTS_ROOT / "results_pca_bert_large_sentences")
DIR_PROM = str(RESULTS_ROOT / "results_pca_bert_large_sentences_prompted")
DIR_CMP  = str(RESULTS_ROOT / "comparison_prompted_vs_unprompted_bert")
os.makedirs(DIR_CMP, exist_ok=True)

ALL_CATS = ["Shakespeare", "Computer Science", "Information Security",
            "Theory of Computation", "Hate Speech", "No-Hate Speech"]
PAIRS = [
    ("Computer Science",      "Shakespeare"),
    ("Theory of Computation", "Information Security"),
    ("Hate Speech",           "No-Hate Speech"),
]
COLORS = {
    "Shakespeare":           "#E06C75",
    "Computer Science":      "#61AFEF",
    "Information Security":  "#98C379",
    "Theory of Computation": "#E5C07B",
    "Hate Speech":           "#C678DD",
    "No-Hate Speech":        "#56B6C2",
}

SILHOUETTE_DELTA = 0.05
SEP_RATIO_DELTA  = 0.10

def load_run(folder):
    emb  = np.vstack([np.load(os.path.join(folder, "embeddings.npy")),
                      np.load(os.path.join(folder, "hate_embeddings.npy"))])
    labs = np.concatenate([np.load(os.path.join(folder, "labels.npy"),      allow_pickle=True),
                           np.load(os.path.join(folder, "hate_labels.npy"), allow_pickle=True)])
    return emb, labs

print("Loading embeddings ...")
emb_u, labs_u = load_run(DIR_UNPR)
emb_p, labs_p = load_run(DIR_PROM)
print(f"  Unprompted: {emb_u.shape}  |  Prompted: {emb_p.shape}")

def to_pca50(emb):
    n = min(50, emb.shape[0] - 1, emb.shape[1])
    pca = PCA(n_components=n)
    return pca.fit_transform(emb), pca

emb_u50, _ = to_pca50(emb_u)
emb_p50, _ = to_pca50(emb_p)

def sil(emb50, labs, cats):
    mask = np.isin(labs, cats)
    return silhouette_score(emb50[mask], labs[mask]) if mask.sum() > 1 else 0.0

def sep_ratio(emb, labs, c1, c2):
    d1, d2 = emb[labs == c1], emb[labs == c2]
    def avg_intra(d):
        n = d.shape[0]; dmat = cdist(d, d, "euclidean"); triu = np.triu_indices(n, k=1)
        return dmat[triu].mean() if len(triu[0]) > 0 else 1.0
    inter = cdist(d1, d2, "euclidean").mean()
    intra = np.mean([avg_intra(d1), avg_intra(d2)])
    return inter / intra if intra > 0 else 0.0

report = ["="*70, "PROMPTED vs UNPROMPTED — BERT-large-uncased", "="*70]
flags  = []

report.append("\n  SILHOUETTE SCORES (PCA-50)")
report.append(f"  {'Scope':<35} {'Unprompted':>12} {'Prompted':>12} {'Delta':>8} {'Flag':>6}")
report.append(f"  {'-'*75}")
for label, cats in [("All 6 categories", ALL_CATS)] + [(f"{c1} vs {c2}", [c1,c2]) for c1,c2 in PAIRS]:
    su = sil(emb_u50, labs_u, cats)
    sp = sil(emb_p50, labs_p, cats)
    delta = sp - su; flagged = abs(delta) >= SILHOUETTE_DELTA
    if flagged: flags.append(f"Silhouette [{label}]: {su:.3f} → {sp:.3f}  (Δ={delta:+.3f})")
    report.append(f"  {label:<35} {su:>12.4f} {sp:>12.4f} {delta:>+8.4f} {'***' if flagged else '':>6}")

report.append("\n  SEPARABILITY RATIO (inter / avg intra, full-d Euclidean)")
report.append(f"  {'Pair':<35} {'Unprompted':>12} {'Prompted':>12} {'Rel Δ':>8} {'Flag':>6}")
report.append(f"  {'-'*75}")
for c1, c2 in PAIRS:
    ru = sep_ratio(emb_u, labs_u, c1, c2)
    rp = sep_ratio(emb_p, labs_p, c1, c2)
    rel_delta = (rp - ru) / ru if ru > 0 else float("inf")
    flagged   = abs(rel_delta) >= SEP_RATIO_DELTA
    if flagged: flags.append(f"Sep ratio [{c1} vs {c2}]: {ru:.3f} → {rp:.3f}  (rel Δ={rel_delta:+.1%})")
    report.append(f"  {f'{c1} vs {c2}':<35} {ru:>12.4f} {rp:>12.4f} {rel_delta:>+8.1%} {'***' if flagged else '':>6}")

report.append(f"\n{'='*70}")
if flags:
    report.append(f"  MAJOR CHANGES DETECTED ({len(flags)}):")
    for f in flags: report.append(f"    *** {f}")
else:
    report.append("  No major changes detected above threshold.")
report.append(f"{'='*70}")

txt = "\n".join(report); print(txt)
with open(os.path.join(DIR_CMP, "comparison_report.txt"), "w", encoding="utf-8") as f: f.write(txt)
print("Saved comparison_report.txt")

for c1, c2 in PAIRS:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, emb50, labs, title_tag in [
        (axes[0], emb_u50, labs_u, "Unprompted"),
        (axes[1], emb_p50, labs_p, "Prompted"),
    ]:
        mask = np.isin(labs, [c1, c2])
        pca2 = PCA(n_components=2); X2 = pca2.fit_transform(emb50[mask]); evr = pca2.explained_variance_ratio_
        lab_sub = labs[mask]
        for cat in [c1, c2]:
            idx = lab_sub == cat
            ax.scatter(X2[idx,0], X2[idx,1], c=COLORS[cat], label=cat,
                       alpha=0.75, edgecolors="w", linewidths=0.5, s=60)
        sil_val = sil(emb50, labs, [c1, c2])
        ax.set_xlabel(f"PC 1  ({evr[0]:.1%})", fontsize=11)
        ax.set_ylabel(f"PC 2  ({evr[1]:.1%})", fontsize=11)
        ax.set_title(f"{title_tag}\nSilhouette = {sil_val:.3f}", fontsize=12, fontweight="bold")
        ax.legend(fontsize=10); ax.grid(True, linestyle="--", alpha=0.4)
    fig.suptitle(f"BERT-large-uncased — {c1} vs {c2}\nPCA-2 on PCA-50 embeddings",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    fpath = os.path.join(DIR_CMP, f"compare_pca_{c1}_{c2}".lower().replace(" ","_") + ".png")
    plt.savefig(fpath, dpi=150, bbox_inches="tight")
    print(f"Saved {fpath}")
    plt.show(); plt.close()
